# 02: Modelling: Baselines to SARIMAX to LightGBM

Narrative companion to `src/models/` and `src/evaluate.py`.

**Task definition.** Day-ahead forecasting: at the end of day *D* predict all 24
hours of day *D+1*. This mirrors how load forecasts feed the day-ahead power
market. Every design decision below follows from that origin/horizon choice.

**Anti-leakage rule.** A feature is admissible only if it is known at the
origin. All lag/rolling features are therefore shifted ≥ 24 h, and the tree
model becomes a *direct* day-ahead forecaster, no recursive feedback of its
own predictions.

> Run the pipeline first: `python -m src.data_ingestion && python -m src.features`.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import TARGET, TEST_WEEKS
from src.data_ingestion import load_processed
from src.features import FEATURES, load_features

df = load_processed()
feats = load_features()
print(f'{feats.shape[0]:,} rows, {len(FEATURES)} features')
feats.head(3)

## 1. Baselines: the bar to clear

* **naive-24h**: tomorrow = today (per hour).
* **seasonal naive-168h**: tomorrow = same day last week.

Load is so strongly autocorrelated that these are hard to beat; any model that
cannot outperform them has no business value.

In [ ]:
from src.models.baselines import naive_24h, seasonal_naive_168h

y = df[TARGET]
test_idx = y.index[-28 * 24:]
for name, fn in [('naive-24h', naive_24h), ('seasonal-naive-168h', seasonal_naive_168h)]:
    p = fn(y, test_idx)
    m = p.notna()
    print(f'{name:>20s}  MAPE(last 28d) = '
          f'{np.mean(np.abs((y[test_idx][m]-p[m])/y[test_idx][m]))*100:.2f}%')

## 2. SARIMAX on daily means with temperature exog

Hourly load has three stacked seasonalities, which a single SARIMA cannot
represent well, so the classical model works at the **daily** level:
weekly seasonal order (s = 7) plus exogenous `temp` and centred `temp²`
(the U-shape). Order is selected by AIC over a small grid, the full ranked
table is persisted to `reports/sarimax_order_selection.csv` for the write-up.

For comparability with hourly models, its daily forecast is disaggregated to
hourly using the train-period hour-of-week profile.

In [ ]:
from src.models import sarimax_model as sm

daily = sm.make_daily(df)
train = daily.iloc[:-TEST_WEEKS * 7]   # never touch the backtest window
order, seasonal = sm.select_order(train[TARGET], sm.exog_matrix(train))
print('AIC-selected order:', order, 'x', seasonal)
res = sm.fit(train[TARGET], sm.exog_matrix(train), order, seasonal)
res.summary()

In [ ]:
pd.read_csv('../reports/sarimax_order_selection.csv').head(8)

## 3. LightGBM: the main model

Gradient-boosted trees on the full feature set: calendar (incl. Spanish
holidays), lags (24/48/168 h), day-ahead-safe rolling statistics, and the
temperature terms. Trees capture the hour×weekday×temperature interactions
that the linear models cannot.

In [ ]:
from src.models.lgbm_model import train_point

cutoff = feats.index.max() - pd.Timedelta(weeks=TEST_WEEKS)
train_f = feats[feats.index <= cutoff]
model = train_point(train_f[FEATURES], train_f[TARGET])

imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
imp.plot.barh(figsize=(8, 7), title='LightGBM feature importance (splits)');

## 4. Prediction intervals: quantile LightGBM

Procurement decisions need *ranges*, not just points. Three additional
LightGBM models trained with pinball loss at α = 0.10 / 0.50 / 0.90 provide a
day-ahead P10-P90 band (sorted post-hoc to guarantee monotonicity).

In [ ]:
from src.models.lgbm_model import train_quantiles, predict_quantiles

q_models = train_quantiles(train_f[FEATURES], train_f[TARGET])
week = feats[(feats.index > cutoff) & (feats.index <= cutoff + pd.Timedelta(days=7))]
band = predict_quantiles(q_models, week[FEATURES])

plt.figure(figsize=(13, 5))
plt.fill_between(band.index, band['p10'], band['p90'], alpha=0.25, label='P10-P90')
plt.plot(week.index, week[TARGET], color='black', lw=1.3, label='Actual')
plt.plot(band.index, band['p50'], color='tab:blue', lw=1.3, label='P50')
plt.legend(); plt.title('Quantile band on the first unseen week');

## 5. Honest evaluation: rolling-origin backtest + Diebold-Mariano

A single train/test split can flatter or punish a model by luck. The proper
protocol (in `src/evaluate.py`) rolls the forecast origin day by day across the
final 12 weeks (~2,000 hourly predictions), refitting LightGBM weekly and
updating SARIMAX's state daily.

Because forecast errors at a 24 h horizon are autocorrelated, "model A's MAPE
is lower" is not enough, the **Diebold-Mariano test** (squared-error loss,
Newey-West variance, Harvey-Leybourne-Newbold small-sample correction) checks
whether the accuracy gap between the top two models is statistically
significant. Results land in `reports/results.md` and the README.

In [ ]:
# Full backtest (a few minutes):
#   python -m src.evaluate
from src.config import REPORTS
res_file = REPORTS / 'results.md'
if res_file.exists():
    from IPython.display import Markdown
    display(Markdown(res_file.read_text()))
else:
    print('Run `python -m src.evaluate` to generate backtest results.')

## Conclusions

* Lag structure dominates: even naive baselines are strong, so honest
  benchmarking matters more than model sophistication.
* SARIMAX is a solid, interpretable statistical reference once temperature
  enters as an exogenous quadratic, but one daily value spread by a fixed
  profile cannot react to hour-level dynamics.
* LightGBM with leakage-safe features wins on every metric; the DM test
  (see README) confirms the gap is statistically significant.
* Quantile models add the P10-P90 band that procurement actually consumes.

Next: `python -m src.forecast` for the operational next-24h forecast, and
`streamlit run app.py` for the interactive dashboard.